# Top-K: BF16 vs FP32 vs Torch

Both kernels receive the same values: BF16 logits for the new variant, promoted to FP32 for the old variant and Torch. Input conversion and allocation are outside timing. Torch runs FP32 activation + top-K + conversion to int32 indices/BF16 weights.

`us` is CUDA-graph replay time per call; `xTorch` is Torch time divided by variant time. `overlap%` measures selected-ID overlap with Torch (ties can differ). `max_abs` and `max_rel%` measure weights against Torch's FP32 scores at the selected IDs. `mass_gap` is the largest per-token loss in selected score sum versus Torch's top-K.

The BF16 tests allow 2% relative selection-score error and 5% weight error; the table reports the actual errors. The legacy FP32 path runs only when `N % 8 == 0`, `E % 256 == 0`, and `K <= E / 32`. BF16 additionally supports 128/384 experts and token tails. Both paths require `K <= 16` and `E <= 1024`.

Run against a checkout containing these changes. A fresh Colab runtime clones the `topk` branch. Restart the notebook kernel if rebuilding an extension already imported in this session.


In [7]:
!rm -r xCaliber

In [8]:
import os
import sys
import subprocess
from pathlib import Path

root = next((p for p in (Path.cwd(), Path.cwd().parent, Path.cwd() / "xCaliber")
             if (p / "xcaliber" / "setup.py").is_file()), None)
if root is None:
    root = Path.cwd() / "xCaliber"
    subprocess.run(["git", "clone", "-b", "fmoe", "https://github.com/Pranshu-Bahadur/xCaliber.git", str(root)], check=True)
root = root.resolve()
print(root)


/content/xCaliber


In [9]:
subprocess.run([sys.executable, "-m", "pip", "install", "ninja", "pytest"], check=True)


CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', 'ninja', 'pytest'], returncode=0)

In [10]:
import torch

assert torch.cuda.is_available(), "Select a CUDA GPU runtime"
os.environ["TORCH_CUDA_ARCH_LIST"] = ".".join(map(str, torch.cuda.get_device_capability()))
os.environ.setdefault("MAX_JOBS", "4")
print(torch.cuda.get_device_name(), "sm", os.environ["TORCH_CUDA_ARCH_LIST"], "CUDA", torch.version.cuda)
subprocess.run([sys.executable, str(root / "xcaliber" / "setup.py"), "build_ext", "--inplace"], cwd=root, check=True)


NVIDIA A100-SXM4-40GB sm 8.0 CUDA 12.8


CompletedProcess(args=['/usr/bin/python3', '/content/xCaliber/xcaliber/setup.py', 'build_ext', '--inplace'], returncode=0)

In [11]:
subprocess.run([sys.executable, "-m", "pytest", str(root / "test" / "test_moe.py"), "-q"], cwd=root, check=True)


CompletedProcess(args=['/usr/bin/python3', '-m', 'pytest', '/content/xCaliber/test/test_moe.py', '-q'], returncode=0)

In [12]:
import importlib.util

sys.path.insert(0, str(root))
spec = importlib.util.spec_from_file_location("test_moe", root / "test" / "test_moe.py")
moe_tests = importlib.util.module_from_spec(spec)
spec.loader.exec_module(moe_tests)


In [13]:
Ns = (8, 16, 16384)
Es = (256, 512)
Ks = (2, 8)
results = moe_tests.benchmark(Ns=Ns, Es=Es, Ks=Ks, repeat=100)


GPU: NVIDIA A100-SXM4-40GB | Torch 2.11.0+cu128 | CUDA 12.8
Speedup = baseline / BF16; >1 means BF16 is faster. '-' means unsupported.

Sigmoid | latency in us
Tokens Experts   K      BF16  Old FP32     Torch  vs FP32 vs Torch
     8     256   2     3.965     4.510    15.102    1.14x    3.81x
     8     256   8     5.501     8.743    15.124    1.59x    2.75x
     8     512   2     6.068     7.115    15.141    1.17x    2.50x
     8     512   8     8.651    14.002    18.055    1.62x    2.09x
    16     256   2     3.998     4.561    14.660    1.14x    3.67x
    16     256   8     5.532     8.860    14.868    1.60x    2.69x
    16     512   2     6.126     7.158    15.380    1.17x    2.51x
    16     512   8     8.712    13.834    18.766    1.59x    2.15x
 16384     256   2    33.343    54.417   383.492    1.63x   11.50x
 16384     256   8    60.281   166.881   406.632    2.77x    6.75x
 16384     512   2    61.055    93.530   650.115    1.53x   10.65x
 16384     512   8   129.157   265.2